In [8]:
# ============================================
# Cell 1: بررسی و Debug خطای numpy.stack (اصلاح شده)
# ============================================

import numpy as np
import sys
sys.path.append('/home/GAN_Writing_Farsi')

print("NumPy version:", np.__version__)
print("="*60)

# بررسی دقیق‌تر loadData
import load_data as ld_en

# Debug mode - ببینیم دقیقاً کجا خطا میده
print("\nشروع لود داده انگلیسی با debug...")

try:
    # اول ببینیم تا کجا پیش میره
    data_train_en, data_test_en = ld_en.loadData(oov=False)
    print("✓ داده‌ها لود شدند")
    print(f"✓ تعداد train: {len(data_train_en)}")
    print(f"✓ تعداد test: {len(data_test_en)}")
    
except Exception as e:
    print(f"✗ خطا: {e}")
    print("\nStack trace کامل:")
    import traceback
    traceback.print_exc()


NumPy version: 1.24.3

شروع لود داده انگلیسی با debug...
✓ داده‌ها لود شدند
✓ تعداد train: 339
✓ تعداد test: 161


In [9]:
# ============================================
# Cell 2: بررسی دستی کلاس IAM_words (اصلاح شده)
# ============================================

import cv2
import os

print("بررسی ساختار dataset...")
print("="*60)

# بررسی data_dict
print(f"Type of data_train_en.data_dict: {type(data_train_en.data_dict)}")
print(f"تعداد کلیدها (writers): {len(data_train_en.data_dict)}")

# نگاه به یک نمونه
first_key = list(data_train_en.data_dict.keys())[0]
print(f"\nاولین writer ID (mapped): {first_key}")
print(f"تعداد کلمات این نویسنده: {len(data_train_en.data_dict[first_key])}")
print(f"نمونه کلمه اول: {data_train_en.data_dict[first_key][0]}")

# تست یک آیتم
try:
    print("\n" + "="*60)
    print("تست آیتم اول...")
    print("="*60)
    
    item = data_train_en[0]  # wid_idx_num = 0
    
    print(f"✓ Type: {type(item)}")
    print(f"✓ Length: {len(item)}")
    
    # بررسی هر قسمت
    mode, wid, idxs, imgs, widths, labels, img_xt, label_xt, label_swap = item
    
    print(f"\n📊 جزئیات:")
    print(f"  mode: {mode}")
    print(f"  wid: {wid}")
    print(f"  idxs type: {type(idxs)}, len: {len(idxs) if hasattr(idxs, '__len__') else 'N/A'}")
    print(f"  imgs type: {type(imgs)}")
    print(f"  imgs shape: {imgs.shape if hasattr(imgs, 'shape') else 'no shape'}")
    print(f"  imgs dtype: {imgs.dtype if hasattr(imgs, 'dtype') else 'N/A'}")
    print(f"  widths type: {type(widths)}")
    print(f"  widths len: {len(widths) if hasattr(widths, '__len__') else 'N/A'}")
    print(f"  labels type: {type(labels)}")
    print(f"  labels shape: {np.array(labels).shape}")
    print(f"  img_xt shape: {img_xt.shape if hasattr(img_xt, 'shape') else 'no shape'}")
    print(f"  label_xt: {label_xt}")
    
    # اینجا مشکل numpy.stack ممکنه باشه
    if hasattr(imgs, 'shape'):
        print(f"\n✓ imgs به درستی یک numpy array است با shape: {imgs.shape}")
        print(f"  Expected shape: ({ld_en.NUM_CHANNEL}, {ld_en.IMG_HEIGHT}, {ld_en.IMG_WIDTH})")
    else:
        print(f"\n✗ imgs یک numpy array نیست! Type: {type(imgs)}")
    
    print("\n✅ آیتم اول با موفقیت لود شد!")
    
except Exception as e:
    print(f"\n✗ خطا در تست آیتم: {e}")
    import traceback
    traceback.print_exc()


بررسی ساختار dataset...
Type of data_train_en.data_dict: <class 'dict'>
تعداد کلیدها (writers): 339

اولین writer ID (mapped): 0
تعداد کلمات این نویسنده: 53
نمونه کلمه اول: ['049,a03-034-00-00', 'Members']

تست آیتم اول...
✓ Type: <class 'tuple'>
✓ Length: 9

📊 جزئیات:
  mode: src
  wid: 0
  idxs type: <class 'numpy.ndarray'>, len: 15
  imgs type: <class 'numpy.ndarray'>
  imgs shape: (15, 64, 216)
  imgs dtype: float32
  widths type: <class 'numpy.ndarray'>
  widths len: 15
  labels type: <class 'numpy.ndarray'>
  labels shape: (15, 12)
  img_xt shape: (1, 64, 216)
  label_xt: [0, 17, 16, 1, 2, 2, 2, 2, 2, 2, 2, 2]

✓ imgs به درستی یک numpy array است با shape: (15, 64, 216)
  Expected shape: (15, 64, 216)

✅ آیتم اول با موفقیت لود شد!


In [10]:
# ============================================
# Cell 3: بررسی دقیق فرآیند __getitem__
# ============================================

print("بررسی دستی فرآیند __getitem__...")
print("="*60)

try:
    # دریافت یک writer
    wid_idx_num = 0
    words = data_train_en.data_dict[wid_idx_num]
    
    print(f"Writer index: {wid_idx_num}")
    print(f"تعداد کلمات: {len(words)}")
    print(f"نمونه کلمه: {words[0]}")
    
    # شبیه‌سازی فرآیند __getitem__
    np.random.shuffle(words)
    
    wids = list()
    idxs = list()
    imgs = list()
    img_widths = list()
    labels = list()
    
    print(f"\nپردازش {min(len(words), ld_en.EXTRA_CHANNEL)} کلمه...")
    
    for i, word in enumerate(words[:ld_en.EXTRA_CHANNEL]):
        wid, idx = word[0].split(",")
        
        # خواندن تصویر
        img_path = os.path.join(ld_en.img_base, idx + '.png')
        
        if not os.path.exists(img_path):
            print(f"  ⚠️  تصویر {i}: وجود ندارد - {img_path}")
            continue
        
        img, img_width = data_train_en.read_image_single(idx)
        label = data_train_en.label_padding(" ".join(word[1:]), ld_en.num_tokens)
        
        print(f"  کلمه {i}:")
        print(f"    wid: {wid}, idx: {idx}")
        print(f"    text: {' '.join(word[1:])}")
        print(f"    img shape: {img.shape}, dtype: {img.dtype}")
        print(f"    img_width: {img_width}")
        print(f"    label: {label[:5]}... (len={len(label)})")
        
        wids.append(wid)
        idxs.append(idx)
        imgs.append(img)
        img_widths.append(img_width)
        labels.append(label)
    
    print(f"\n📊 آماده سازی برای stack:")
    print(f"  تعداد تصاویر جمع‌آوری شده: {len(imgs)}")
    print(f"  NUM_CHANNEL: {ld_en.NUM_CHANNEL}")
    print(f"  EXTRA_CHANNEL: {ld_en.EXTRA_CHANNEL}")
    
    # بررسی کنیم همه تصاویر هم‌اندازه هستند؟
    shapes = [img.shape for img in imgs]
    print(f"\n  Shapes (اول تا آخر):")
    for i, shape in enumerate(shapes):
        print(f"    {i}: {shape}")
    
    unique_shapes = set(shapes)
    if len(unique_shapes) > 1:
        print(f"\n  ⚠️  تصاویر اندازه‌های مختلف دارند!")
        print(f"  Unique shapes: {unique_shapes}")
    else:
        print(f"\n  ✓ همه تصاویر هم‌اندازه هستند: {shapes[0]}")
    
    # حالا تلاش برای stack
    print(f"\n  تلاش برای stack {len(imgs)} تصویر...")
    
    num_imgs = len(imgs)
    if num_imgs >= ld_en.EXTRA_CHANNEL:
        print(f"  حالت 1: num_imgs ({num_imgs}) >= EXTRA_CHANNEL ({ld_en.EXTRA_CHANNEL})")
        final_img = np.stack(imgs[:ld_en.EXTRA_CHANNEL], axis=0)
        print(f"  ✓ Stack موفق! Shape: {final_img.shape}")
    else:
        print(f"  حالت 2: num_imgs ({num_imgs}) < EXTRA_CHANNEL ({ld_en.EXTRA_CHANNEL})")
        print(f"  نیاز به تکرار...")
        
        final_img_list = imgs.copy()
        while len(final_img_list) < ld_en.EXTRA_CHANNEL:
            num_cp = ld_en.EXTRA_CHANNEL - len(final_img_list)
            print(f"    کپی {num_cp} تصویر...")
            final_img_list = final_img_list + imgs[:num_cp]
        
        print(f"  تعداد نهایی قبل از stack: {len(final_img_list)}")
        final_img = np.stack(final_img_list, axis=0)
        print(f"  ✓ Stack موفق! Shape: {final_img.shape}")
    
    print(f"\n✅ فرآیند دستی با موفقیت انجام شد!")
    
except Exception as e:
    print(f"\n✗ خطا در فرآیند دستی: {e}")
    import traceback
    traceback.print_exc()


بررسی دستی فرآیند __getitem__...
Writer index: 0
تعداد کلمات: 53
نمونه کلمه: ['049,a03-034-06-08', 'of']

پردازش 16 کلمه...
  کلمه 0:
    wid: 049, idx: a03-034-06-03
    text: has
    img shape: (64, 216), dtype: float32
    img_width: 145
    label: [0, 10, 3, 21, 1]... (len=12)
  کلمه 1:
    wid: 049, idx: a03-034-04-06
    text: there
    img shape: (64, 216), dtype: float32
    img_width: 168
    label: [0, 22, 10, 7, 20]... (len=12)
  کلمه 2:
    wid: 049, idx: a03-034-01-03
    text: booklet
    img shape: (64, 216), dtype: float32
    img_width: 216
    label: [0, 4, 17, 17, 13]... (len=12)
  کلمه 3:
    wid: 049, idx: a03-034-06-07
    text: face
    img shape: (64, 216), dtype: float32
    img_width: 105
    label: [0, 8, 3, 5, 7]... (len=12)
  کلمه 4:
    wid: 049, idx: a03-034-06-04
    text: fallen
    img shape: (64, 216), dtype: float32
    img_width: 114
    label: [0, 8, 3, 14, 14]... (len=12)
  کلمه 5:
    wid: 049, idx: a03-034-05-02
    text: the
    img shape: (64,

In [11]:
# ============================================
# Cell 4: تست چند آیتم متوالی
# ============================================

print("تست لود چند آیتم متوالی...")
print("="*60)

num_test = min(5, len(data_train_en))

for i in range(num_test):
    try:
        print(f"\nآیتم {i}:")
        item = data_train_en[i]
        mode, wid, idxs, imgs, widths, labels, img_xt, label_xt, label_swap = item
        
        print(f"  ✓ wid: {wid}")
        print(f"  ✓ imgs shape: {imgs.shape}")
        print(f"  ✓ img_xt shape: {img_xt.shape}")
        print(f"  ✓ تعداد labels: {len(labels)}")
        
    except Exception as e:
        print(f"  ✗ خطا در آیتم {i}: {e}")
        import traceback
        traceback.print_exc()
        break

print(f"\n{'='*60}")
if i == num_test - 1:
    print("✅ همه آیتم‌ها با موفقیت لود شدند!")
else:
    print(f"⚠️  خطا در آیتم {i}")


تست لود چند آیتم متوالی...

آیتم 0:
  ✓ wid: 0
  ✓ imgs shape: (15, 64, 216)
  ✓ img_xt shape: (1, 64, 216)
  ✓ تعداد labels: 15

آیتم 1:
  ✓ wid: 1
  ✓ imgs shape: (15, 64, 216)
  ✓ img_xt shape: (1, 64, 216)
  ✓ تعداد labels: 15

آیتم 2:
  ✓ wid: 2
  ✓ imgs shape: (15, 64, 216)
  ✓ img_xt shape: (1, 64, 216)
  ✓ تعداد labels: 15

آیتم 3:
  ✓ wid: 3
  ✓ imgs shape: (15, 64, 216)
  ✓ img_xt shape: (1, 64, 216)
  ✓ تعداد labels: 15

آیتم 4:
  ✓ wid: 4
  ✓ imgs shape: (15, 64, 216)
  ✓ img_xt shape: (1, 64, 216)
  ✓ تعداد labels: 15

✅ همه آیتم‌ها با موفقیت لود شدند!


In [12]:
# ============================================
# Cell 5: بررسی کد منبع __getitem__
# ============================================

import inspect

print("کد منبع __getitem__ در IAM_words:")
print("="*60)

source = inspect.getsource(ld_en.IAM_words.__getitem__)
print(source)


کد منبع __getitem__ در IAM_words:
    def __getitem__(self, wid_idx_num):
        # print("wid_idx_num: ", wid_idx_num)
        words = self.data_dict[wid_idx_num]
        """shuffle images"""
        np.random.shuffle(words)

        wids = list()
        idxs = list()
        imgs = list()
        img_widths = list()
        labels = list()

        for word in words:
            wid, idx = word[0].split(",")
            img, img_width = self.read_image_single(idx)
            label = self.label_padding(" ".join(word[1:]), num_tokens)
            wids.append(wid)
            idxs.append(idx)
            imgs.append(img)
            img_widths.append(img_width)
            labels.append(label)

        if len(list(set(wids))) != 1:
            print("Error! writer id differs")
            exit()

        final_wid = wid_idx_num
        num_imgs = len(imgs)
        if num_imgs >= EXTRA_CHANNEL:
            final_img = np.stack(imgs[:EXTRA_CHANNEL], axis=0)  # 64, h, w
            final

In [13]:
# Restart kernel و اجرای این سلول

import sys
sys.path.insert(0, './code')

from load_data_farsi import loadData as loadData_farsi

print("تست دیتالودر فارسی (ABAN):")
print("="*60)

data_train_fa, data_test_fa = loadData_farsi(oov=False)

print(f"\n✓ تعداد train: {len(data_train_fa)}")
print(f"✓ تعداد test: {len(data_test_fa)}")

print("\nتست لود یک نمونه:")
try:
    item_fa = data_train_fa[0]
    print(f"✅ لود موفق!")
    print(f"   Writer ID: {item_fa[1]}")
    print(f"   Images shape: {item_fa[3].shape}")
    print(f"   Labels shape: {item_fa[5].shape}")
except Exception as e:
    print(f"✗ خطا: {e}")
    import traceback
    traceback.print_exc()


تست دیتالودر فارسی (ABAN):
تعداد نویسندگان train: 350
تعداد نویسندگان test: 150

✓ تعداد train: 350
✓ تعداد test: 150

تست لود یک نمونه:
✗ خطا: 'S'


Traceback (most recent call last):
  File "/tmp/ipykernel_188119/2192502977.py", line 18, in <module>
    item_fa = data_train_fa[0]
  File "/home/GAN_Writing_Farsi/load_data_farsi.py", line 96, in __getitem__
    label = self.label_padding(" ".join(word[1:]), num_tokens)
  File "/home/GAN_Writing_Farsi/load_data_farsi.py", line 194, in label_padding
    ll = [letter2index[i] for i in labels]
  File "/home/GAN_Writing_Farsi/load_data_farsi.py", line 194, in <listcomp>
    ll = [letter2index[i] for i in labels]
KeyError: 'S'


In [14]:
# Restart kernel و اجرای این سلول

import sys
sys.path.insert(0, './code')

from load_data_farsi import loadData as loadData_farsi

print("تست دیتالودر فارسی (ABAN):")
print("="*60)

data_train_fa, data_test_fa = loadData_farsi(oov=False)

print(f"\n✓ تعداد train: {len(data_train_fa)}")
print(f"✓ تعداد test: {len(data_test_fa)}")

print("\nتست لود یک نمونه:")
try:
    item_fa = data_train_fa[0]
    print(f"✅ لود موفق!")
    print(f"   Writer ID: {item_fa[1]}")
    print(f"   Images shape: {item_fa[3].shape}")
    print(f"   Labels shape: {item_fa[5].shape}")
except Exception as e:
    print(f"✗ خطا: {e}")
    import traceback
    traceback.print_exc()


تست دیتالودر فارسی (ABAN):
تعداد نویسندگان train: 350
تعداد نویسندگان test: 150

✓ تعداد train: 350
✓ تعداد test: 150

تست لود یک نمونه:
✗ خطا: 'I'


Traceback (most recent call last):
  File "/tmp/ipykernel_188119/2192502977.py", line 18, in <module>
    item_fa = data_train_fa[0]
  File "/home/GAN_Writing_Farsi/load_data_farsi.py", line 96, in __getitem__
    label = self.label_padding(" ".join(word[1:]), num_tokens)
  File "/home/GAN_Writing_Farsi/load_data_farsi.py", line 194, in label_padding
    ll = [letter2index[i] for i in labels]
  File "/home/GAN_Writing_Farsi/load_data_farsi.py", line 194, in <listcomp>
    ll = [letter2index[i] for i in labels]
KeyError: 'I'


In [15]:
# بررسی کاراکترهای موجود در groundtruth

import sys
sys.path.insert(0, './code')

src = "Groundtruth_farsi/gan.aban.tr_va.gt.filter27"
tar = "Groundtruth_farsi/gan.aban.test.gt.filter27"

def analyze_groundtruth(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    all_chars = set()
    sample_lines = []
    
    for i, line in enumerate(lines):
        parts = line.strip().split(' ')
        if len(parts) > 1:
            # parts[0] = wid,idx
            # parts[1:] = کلمه
            word = ' '.join(parts[1:])
            all_chars.update(word)
            if i < 5:  # نمونه
                sample_lines.append((parts[0], word))
    
    return all_chars, sample_lines

print("تحلیل فایل Train:")
print("="*60)
chars_tr, samples_tr = analyze_groundtruth(src)
print(f"تعداد کاراکترهای یونیک: {len(chars_tr)}")
print(f"کاراکترها: {sorted(chars_tr)}")
print(f"\nنمونه‌های اول:")
for wid_idx, word in samples_tr:
    print(f"  {wid_idx}: '{word}'")

print("\n" + "="*60)
print("تحلیل فایل Test:")
print("="*60)
chars_te, samples_te = analyze_groundtruth(tar)
print(f"تعداد کاراکترهای یونیک: {len(chars_te)}")
print(f"کاراکترها: {sorted(chars_te)}")
print(f"\nنمونه‌های اول:")
for wid_idx, word in samples_te:
    print(f"  {wid_idx}: '{word}'")

print("\n" + "="*60)
print("مقایسه با الفبای تعریف شده:")
print("="*60)
defined_chars = set("آابپتثجچحخدذرزژسشصضطظعغفقکگلمنوهیئء ‌")
print(f"الفبای تعریف شده: {sorted(defined_chars)}")

all_gt_chars = chars_tr | chars_te
missing_in_alphabet = all_gt_chars - defined_chars
extra_in_alphabet = defined_chars - all_gt_chars

if missing_in_alphabet:
    print(f"\n⚠️  کاراکترهای موجود در GT که در الفبا نیستند: {sorted(missing_in_alphabet)}")
if extra_in_alphabet:
    print(f"\n✓ کاراکترهای اضافی در الفبا: {sorted(extra_in_alphabet)}")


تحلیل فایل Train:
تعداد کاراکترهای یونیک: 27
کاراکترها: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '_']

نمونه‌های اول:
  0000476,ID0000476_p4_B43: 'TOMAN'
  0000476,ID0000476_p4_B52: 'THOUSAND'
  0000476,ID0000476_p5_B45: 'SEVENHUNDRED'
  0000476,ID0000476_p4_B40: 'TEL'
  0000476,ID0000476_p5_B12: 'NINETEENTH'

تحلیل فایل Test:
تعداد کاراکترهای یونیک: 27
کاراکترها: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '_']

نمونه‌های اول:
  0000524,ID0000524_p4_B43: 'TOMAN'
  0000524,ID0000524_p4_B52: 'THOUSAND'
  0000524,ID0000524_p5_B45: 'SEVENHUNDRED'
  0000524,ID0000524_p4_B40: 'TEL'
  0000524,ID0000524_p5_B12: 'NINETEENTH'

مقایسه با الفبای تعریف شده:
الفبای تعریف شده: [' ', 'ء', 'آ', 'ئ', 'ا', 'ب', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف', 'ق', 'ل', 'م', 'ن', 'ه', 'و', 

In [18]:
import os

# بررسی نوع
gt_file = 'Groundtruth_farsi/gan.aban.tr_va.gt.filter27'

print(f"آیا فایل است؟ {os.path.isfile(gt_file)}")
print(f"آیا دایرکتوری است؟ {os.path.isdir(gt_file)}")
print(f"اندازه: {os.path.getsize(gt_file) if os.path.exists(gt_file) else 'وجود نداره'} بایت")

# خواندن چند خط اول برای دیدن فرمت
print("\n📄 چند خط اول فایل:")
with open(gt_file, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 10:
            print(f"خط {i+1}: {line.strip()}")
        else:
            break


آیا فایل است؟ True
آیا دایرکتوری است؟ False
اندازه: 1424243 بایت

📄 چند خط اول فایل:
خط 1: 0000476,ID0000476_p4_B43 TOMAN
خط 2: 0000476,ID0000476_p4_B52 THOUSAND
خط 3: 0000476,ID0000476_p5_B45 SEVENHUNDRED
خط 4: 0000476,ID0000476_p4_B40 TEL
خط 5: 0000476,ID0000476_p5_B12 NINETEENTH
خط 6: 0000476,ID0000476_p5_B41 SEVENTY
خط 7: 0000476,ID0000476_p6_B14 ORDIBEHESHT
خط 8: 0000476,ID0000476_p4_B49 FIRST
خط 9: 0000476,ID0000476_p6_B18 ESFAND
خط 10: 0000476,ID0000476_p6_B21 AZAR


In [19]:
# استخراج همه کاراکترهای یونیک
all_chars = set()
all_labels = []
total_lines = 0

with open(gt_file, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            total_lines += 1
            # فرمت معمول: filename label
            parts = line.split()
            if len(parts) >= 2:
                label = parts[1]  # قسمت label
                all_labels.append(label)
                all_chars.update(label)

print(f"\n✅ تعداد کل خطوط: {total_lines}")
print(f"✅ کاراکترهای یونیک: {sorted(all_chars)}")
print(f"✅ تعداد کاراکترها: {len(all_chars)}")

# نمایش با جزئیات بیشتر
print(f"\n🔤 لیست کاراکترها:")
for char in sorted(all_chars):
    print(f"  '{char}' (Unicode: U+{ord(char):04X}, نام: {repr(char)})")

# چند نمونه label
print(f"\n📝 نمونه labels:")
for label in all_labels[:10]:
    print(f"  {label}")



✅ تعداد کل خطوط: 43745
✅ کاراکترهای یونیک: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '_']
✅ تعداد کاراکترها: 27

🔤 لیست کاراکترها:
  'A' (Unicode: U+0041, نام: 'A')
  'B' (Unicode: U+0042, نام: 'B')
  'C' (Unicode: U+0043, نام: 'C')
  'D' (Unicode: U+0044, نام: 'D')
  'E' (Unicode: U+0045, نام: 'E')
  'F' (Unicode: U+0046, نام: 'F')
  'G' (Unicode: U+0047, نام: 'G')
  'H' (Unicode: U+0048, نام: 'H')
  'I' (Unicode: U+0049, نام: 'I')
  'J' (Unicode: U+004A, نام: 'J')
  'K' (Unicode: U+004B, نام: 'K')
  'L' (Unicode: U+004C, نام: 'L')
  'M' (Unicode: U+004D, نام: 'M')
  'N' (Unicode: U+004E, نام: 'N')
  'O' (Unicode: U+004F, نام: 'O')
  'P' (Unicode: U+0050, نام: 'P')
  'Q' (Unicode: U+0051, نام: 'Q')
  'R' (Unicode: U+0052, نام: 'R')
  'S' (Unicode: U+0053, نام: 'S')
  'T' (Unicode: U+0054, نام: 'T')
  'U' (Unicode: U+0055, نام: 'U')
  'V' (Unicode: U+0056, نام: 'V')
  'W' (Unicode: U+0057, نام: 'W

In [ ]:
# فایل: load_data_farsi.py (نسخه نهایی)

import pandas as pd

# ============================================
# 1️⃣ تعریف الفبا (برای مدل - انگلیسی)
# ============================================
labelDictionary = "ABCDEFGHIJKLMNOPQRSTUVWXYZ_"

letter2index = {label: n for n, label in enumerate(labelDictionary)}
index2letter = {v: k for k, v in letter2index.items()}

num_classes = len(labelDictionary)

print(f"✅ الفبای مدل: {num_classes} کاراکتر")
print(f"   {labelDictionary}")

# ============================================
# 2️⃣ بارگذاری نگاشت فارسی ↔ انگلیسی
# ============================================
try:
    # فرمت CSV: ImgName,EnglishLabel,PersianWord
    # مثال: p6_B8,ABAN,آبان
    translation_df = pd.read_csv(
        'word_labels_with_translations.csv',
        encoding='utf-8',
        names=['ImgName', 'EnglishLabel', 'PersianWord']
    )
    
    # ساخت دیکشنری‌های نگاشت
    farsi_to_english = dict(zip(
        translation_df['PersianWord'], 
        translation_df['EnglishLabel']
    ))
    
    english_to_farsi = dict(zip(
        translation_df['EnglishLabel'], 
        translation_df['PersianWord']
    ))
    
    print(f"✅ نگاشت فارسی↔انگلیسی: {len(farsi_to_english)} کلمه")
    
except Exception as e:
    print(f"⚠️ خطا در بارگذاری CSV: {e}")
    farsi_to_english = {}
    english_to_farsi = {}

# ============================================
# 3️⃣ توابع کمکی برای تبدیل
# ============================================

def label_padding(labels, num_writer):
    """
    تبدیل لیست کلمات انگلیسی به ماتریس اعداد
    
    Args:
        labels: لیست کلمات انگلیسی مثل ['ABAN', 'TOMAN']
        num_writer: تعداد نویسنده‌ها
    
    Returns:
        new_label: ماتریس [num_writer, max_len] از ایندکس‌ها
    """
    new_label = []
    label_len = [len(i) for i in labels]
    
    for i in labels:
        tmp_label = [letter2index[c] for c in i]  # تبدیل به ایندکس
        tmp_label.extend([num_classes-1] * (max(label_len) - len(i)))  # padding با '_'
        new_label.append(tmp_label)
    
    return new_label

def label_unpadding_and_translate(label_indices, to_farsi=False):
    """
    تبدیل ماتریس اعداد به کلمات
    
    Args:
        label_indices: ماتریس اعداد
        to_farsi: اگر True باشد، خروجی فارسی برمی‌گردونه
    
    Returns:
        لیست کلمات (انگلیسی یا فارسی)
    """
    words = []
    for indices in label_indices:
        # حذف padding
        chars = [index2letter[idx] for idx in indices if idx != num_classes-1]
        english_word = ''.join(chars)
        
        if to_farsi and english_word in english_to_farsi:
            words.append(english_to_farsi[english_word])
        else:
            words.append(english_word)
    
    return words

# ============================================
# 4️⃣ کلاس دیتاست (مشابه قبل)
# ============================================

import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms
import os

class Persian_words(Dataset):
    def __init__(self, data_dir='./Datasets_ABAN/Words', gt_file='./Groundtruth_farsi/gan.aban.tr_va.gt.filter27'):
        super(Persian_words, self).__init__()
        
        self.data_dir = data_dir
        
        # بارگذاری GT
        with open(gt_file, 'r', encoding='utf-8') as f:
            lines = [line.strip() for line in f if line.strip()]
        
        # ساخت دیکشنری داده‌ها
        temp_dict = {}
        for line in lines:
            parts = line.split()
            if len(parts) >= 2:
                img_name = parts[0]
                label = parts[1]  # برچسب انگلیسی مثل "ABAN"
                
                # استخراج Writer ID (مثلاً از p6_B8_w3)
                writer_id = '_'.join(img_name.split('_')[:2])  # p6_B8
                
                if writer_id not in temp_dict:
                    temp_dict[writer_id] = {'images': [], 'labels': []}
                
                temp_dict[writer_id]['images'].append(img_name)
                temp_dict[writer_id]['labels'].append(label)
        
        # تبدیل به ساختار با کلید عددی
        self.wid_list = sorted(temp_dict.keys())
        self.data_dict = {i: temp_dict[wid] for i, wid in enumerate(self.wid_list)}
        
        self.num_writers = len(self.wid_list)
        
        # Transform برای تصاویر
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((64, 256)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])
        
        print(f"✅ بارگذاری {len(lines)} نمونه از {self.num_writers} نویسنده")
    
    def __len__(self):
        return self.num_writers
    
    def __getitem__(self, wid_idx_num):
        """
        Args:
            wid_idx_num: ایندکس عددی نویسنده (0, 1, 2, ...)
        
        Returns:
            imgs: تصاویر نویسنده
            label: برچسب‌های encode شده (اعداد)
            final_wid: Writer ID واقعی (رشته‌ای)
        """
        writer_data = self.data_dict[wid_idx_num]
        final_wid = self.wid_list[wid_idx_num]
        
        imgs = []
        for img_name in writer_data['images']:
            img_path = os.path.join(self.data_dir, f"{img_name}.png")
            if os.path.exists(img_path):
                img = Image.open(img_path)
                img = self.transform(img)
                imgs.append(img)
        
        imgs = torch.stack(imgs) if imgs else torch.zeros(1, 1, 64, 256)
        
        # Encode کردن labels
        label = label_padding(writer_data['labels'], self.num_writers)
        label = torch.LongTensor(label)
        
        return imgs, label, final_wid

# ============================================
# 5️⃣ تست
# ============================================

if __name__ == '__main__':
    print("\n" + "="*50)
    print("🧪 تست دیتالودر فارسی")
    print("="*50)
    
    dataset = Persian_words()
    
    # تست یک نمونه
    imgs, labels, wid = dataset[0]
    
    print(f"\n📊 نمونه اول:")
    print(f"  Writer ID: {wid}")
    print(f"  تعداد تصاویر: {imgs.shape[0]}")
    print(f"  شکل labels: {labels.shape}")
    print(f"  Labels (encoded): {labels[0].tolist()}")
    
    # Decode به انگلیسی
    english_words = label_unpadding_and_translate([labels[0].tolist()], to_farsi=False)
    print(f"  کلمه انگلیسی: {english_words[0]}")
    
    # Decode به فارسی (اگر mapping موجود باشه)
    farsi_words = label_unpadding_and_translate([labels[0].tolist()], to_farsi=True)
    print(f"  کلمه فارسی: {farsi_words[0]}")


✅ الفبای مدل: 27 کاراکتر
   ABCDEFGHIJKLMNOPQRSTUVWXYZ_
✅ نگاشت فارسی↔انگلیسی: 125 کلمه

🧪 تست دیتالودر فارسی
✅ بارگذاری 43745 نمونه از 1050 نویسنده

📊 نمونه اول:
  Writer ID: 00001,ID00001_p4
  تعداد تصاویر: 1
  شکل labels: torch.Size([42, 11])
  Labels (encoded): [19, 14, 12, 0, 13, 26, 26, 26, 26, 26, 26]
  کلمه انگلیسی: TOMAN
  کلمه فارسی: تومان


In [30]:
# فایل: load_data_farsi.py

import os
import torch.utils.data as D
import random
import string
import cv2
import numpy as np

# ==================== تنظیمات ====================
IMG_HEIGHT = 64
IMG_WIDTH = 216
MAX_CHARS = 10
NUM_CHANNEL = 15
EXTRA_CHANNEL = NUM_CHANNEL + 1
NUM_WRITERS = 500  # فارسی
NORMAL = True
OUTPUT_MAX_LEN = MAX_CHARS + 2  # <GO>+groundtruth+<END>

# ==================== مسیرها ====================
img_base = "./datasets/aban/words"
src = "Groundtruth_farsi/gan.aban.tr_va.gt.filter27"
tar = "Groundtruth_farsi/gan.aban.test.gt.filter27"
text_corpus = "./corpora_farsi/farhang-farsi.tr"  # corpus فارسی

# ==================== الفبا (با _) ====================
def labelDictionary():
    # ⭐ دقیقاً مثل کد اصلی: a-z + A-Z + _
    labels = list(string.ascii_uppercase + string.ascii_lowercase + "_")
    letter2index = {label: n for n, label in enumerate(labels)}
    index2letter = {v: k for k, v in letter2index.items()}
    return len(labels), letter2index, index2letter

num_classes, letter2index, index2letter = labelDictionary()
tokens = {"GO_TOKEN": 0, "END_TOKEN": 1, "PAD_TOKEN": 2}
num_tokens = len(tokens.keys())
vocab_size = num_classes + num_tokens

print(f"🔤 الفبا: {num_classes} کاراکتر")
print(f"   A-Z: 26 حرف بزرگ")
print(f"   a-z: 26 حرف کوچک")
print(f"   _  : 1 underscore")
print(f"   مجموع: {26+26+1} = {num_classes}")

# ==================== بارگذاری Corpus فارسی ====================
def load_farsi_corpus(corpus_path, min_len=3, max_len=MAX_CHARS):
    """
    بارگذاری کلمات فارسی (transliterated به انگلیسی)
    فرمت: هر خط یک کلمه انگلیسی
    """
    if not os.path.exists(corpus_path):
        print(f"⚠️ فایل corpus پیدا نشد: {corpus_path}")
        return []
    
    with open(corpus_path, 'r', encoding='utf-8') as f:
        words = [line.strip().upper() for line in f if line.strip()]
    
    # فیلتر کلمات با طول مناسب و فقط حروف انگلیسی + _
    valid_chars = set(string.ascii_uppercase + "_")
    words = [
        w for w in words 
        if min_len <= len(w) <= max_len and all(c in valid_chars for c in w)
    ]
    
    print(f"✅ بارگذاری corpus: {len(words)} کلمه")
    return words

# بارگذاری corpus
FARSI_CORPUS = load_farsi_corpus(text_corpus)

# ==================== Edit Distance ====================
def edits1(word, min_len=2, max_len=MAX_CHARS):
    "All edits that are one edit away from `word`."
    letters = list(string.ascii_lowercase + "_")  # ⭐ اضافه کردن _
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    deletes = [L + R[1:] for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]
    replaces = [L + c + R[1:] for L, R in splits if R for c in letters]
    inserts = [L + c + R for L, R in splits for c in letters]
    if len(word) <= min_len:
        return random.choice(list(set(transposes + replaces + inserts)))
    elif len(word) >= max_len:
        return random.choice(list(set(deletes + transposes + replaces)))
    else:
        return random.choice(list(set(deletes + transposes + replaces + inserts)))

# ==================== کلاس دیتاست فارسی ====================
class Persian_words(D.Dataset):
    def __init__(self, data_dict, oov, corpus):
        self.data_dict = data_dict
        self.oov = oov
        self.corpus = corpus
        self.output_max_len = OUTPUT_MAX_LEN

    def new_ed1(self, word_ori):
        """
        ایجاد نسخه edit distance 1 از کلمه
        """
        word = np.array(word_ori)  # ⭐ کپی به صورت array
        
        # پیدا کردن موقعیت GO و END
        start_positions = np.where(word == tokens["GO_TOKEN"])[0]
        end_positions = np.where(word == tokens["END_TOKEN"])[0]
        
        if len(start_positions) == 0 or len(end_positions) == 0:
            # اگر توکن‌ها پیدا نشدن، برگردون همون
            return word
        
        start = start_positions[0]
        fin = end_positions[0]
        
        # استخراج کلمه
        word_indices = word[start + 1 : fin]
        word_str = "".join([index2letter[i - num_tokens] for i in word_indices])
        
        # ایجاد نسخه edit
        new_word = edits1(word_str)
        label = np.array(self.label_padding(new_word, num_tokens))
        
        return label

    def __getitem__(self, wid_idx_num):
        words = self.data_dict[wid_idx_num]
        """shuffle images"""
        np.random.shuffle(words)

        wids = list()
        idxs = list()
        imgs = list()
        img_widths = list()
        labels = list()

        for word in words:
            wid, idx = word[0].split(",")
            img, img_width = self.read_image_single(idx)
            
            # ⭐ فقط کلمه (بدون فاصله)
            word_text = "".join(word[1:])
            label = self.label_padding(word_text, num_tokens)
            
            wids.append(wid)
            idxs.append(idx)
            imgs.append(img)
            img_widths.append(img_width)
            labels.append(label)

        if len(list(set(wids))) != 1:
            print("Error! writer id differs")
            exit()

        final_wid = wid_idx_num
        num_imgs = len(imgs)
        if num_imgs >= EXTRA_CHANNEL:
            final_img = np.stack(imgs[:EXTRA_CHANNEL], axis=0)
            final_idx = idxs[:EXTRA_CHANNEL]
            final_img_width = img_widths[:EXTRA_CHANNEL]
            final_label = labels[:EXTRA_CHANNEL]
        else:
            final_idx = idxs
            final_img = imgs
            final_img_width = img_widths
            final_label = labels

            while len(final_img) < EXTRA_CHANNEL:
                num_cp = EXTRA_CHANNEL - len(final_img)
                final_idx = final_idx + idxs[:num_cp]
                final_img = final_img + imgs[:num_cp]
                final_img_width = final_img_width + img_widths[:num_cp]
                final_label = final_label + labels[:num_cp]
            final_img = np.stack(final_img, axis=0)

        # ⭐ تبدیل final_label به آرایه NumPy
        final_label = np.array(final_label)
        final_idx = np.array(final_idx)
        final_img_width = np.array(final_img_width)

        _id = np.random.randint(EXTRA_CHANNEL)
        img_xt = final_img[_id : _id + 1]
        
        if self.oov:
            # استفاده از corpus فارسی برای OOV
            if len(self.corpus) > 0:
                label_xt = random.choice(self.corpus)
                label_xt = np.array(self.label_padding(label_xt, num_tokens))
                
                label_xt_swap = random.choice(self.corpus)
                label_xt_swap = np.array(self.label_padding(label_xt_swap, num_tokens))
            else:
                # fallback: تولید تصادفی
                label_xt = np.random.choice(list(letter2index.keys()), size=random.randint(3, MAX_CHARS))
                label_xt = "".join(label_xt)
                label_xt = np.array(self.label_padding(label_xt, num_tokens))
                
                label_xt_swap = np.random.choice(list(letter2index.keys()), size=random.randint(3, MAX_CHARS))
                label_xt_swap = "".join(label_xt_swap)
                label_xt_swap = np.array(self.label_padding(label_xt_swap, num_tokens))
        else:
            label_xt = final_label[_id]
            label_xt_swap = self.new_ed1(label_xt)

        final_idx = np.delete(final_idx, _id, axis=0)
        final_img = np.delete(final_img, _id, axis=0)
        final_img_width = np.delete(final_img_width, _id, axis=0)
        final_label = np.delete(final_label, _id, axis=0)

        return (
            "src",
            final_wid,
            final_idx,
            final_img,
            final_img_width,
            final_label,
            img_xt,
            label_xt,
            label_xt_swap,
        )

    def __len__(self):
        return len(self.data_dict)

    def read_image_single(self, file_name):
        url = os.path.join(img_base, file_name + ".png")
        img = cv2.imread(url, 0)

        if img is None:
            return np.zeros((IMG_HEIGHT, IMG_WIDTH)), 0

        rate = float(IMG_HEIGHT) / img.shape[0]
        img = cv2.resize(
            img,
            (int(img.shape[1] * rate) + 1, IMG_HEIGHT),
            interpolation=cv2.INTER_CUBIC,
        )
        img = img / 255.0

        img = 1.0 - img
        img_width = img.shape[-1]

        if img_width > IMG_WIDTH:
            outImg = img[:, :IMG_WIDTH]
            img_width = IMG_WIDTH
        else:
            outImg = np.zeros((IMG_HEIGHT, IMG_WIDTH), dtype="float32")
            outImg[:, :img_width] = img
        outImg = outImg.astype("float32")

        mean = 0.5
        std = 0.5
        outImgFinal = (outImg - mean) / std
        return outImgFinal, img_width

    def label_padding(self, labels, num_tokens):
        """
        تبدیل رشته به لیست اعداد با padding
        """
        # ⭐ حذف فاصله‌ها
        labels = labels.replace(" ", "")
        
        ll = [letter2index[i] for i in labels]
        ll = np.array(ll) + num_tokens
        ll = list(ll)
        ll = [tokens["GO_TOKEN"]] + ll + [tokens["END_TOKEN"]]
        
        # padding
        num = self.output_max_len - len(ll)
        if num > 0:
            ll.extend([tokens["PAD_TOKEN"]] * num)
        elif num < 0:
            # اگر خیلی بلند بود، کات کن
            ll = ll[:self.output_max_len - 1] + [tokens["END_TOKEN"]]
        
        return ll

# ==================== بارگذاری داده ====================
def loadData(oov=False):
    gt_tr = src
    gt_te = tar

    # Train
    with open(gt_tr, "r", encoding='utf-8') as f_tr:
        data_tr = f_tr.readlines()
        data_tr = [i.strip().split(" ") for i in data_tr]
        tr_dict = dict()
        
        for i in data_tr:
            parts = i[0].split(",")
            if len(parts) >= 2:
                wid = parts[0]
                idx = parts[1]
                
                if os.path.exists(os.path.join(img_base, idx + ".png")):
                    entry = [f"{wid},{idx}"] + i[1:]
                    
                    if wid not in tr_dict.keys():
                        tr_dict[wid] = [entry]
                    else:
                        tr_dict[wid].append(entry)
        
        wid2label_tr = {wid: idx for idx, wid in enumerate(sorted(tr_dict.keys()))}
        
        new_tr_dict = dict()
        for k in tr_dict.keys():
            new_tr_dict[wid2label_tr[k]] = tr_dict[k]
    
    # Test
    with open(gt_te, "r", encoding='utf-8') as f_te:
        data_te = f_te.readlines()
        data_te = [i.strip().split(" ") for i in data_te]
        te_dict = dict()
        
        for i in data_te:
            parts = i[0].split(",")
            if len(parts) >= 2:
                wid = parts[0]
                idx = parts[1]
                
                if os.path.exists(os.path.join(img_base, idx + ".png")):
                    entry = [f"{wid},{idx}"] + i[1:]
                    
                    if wid not in te_dict.keys():
                        te_dict[wid] = [entry]
                    else:
                        te_dict[wid].append(entry)
        
        wid2label_te = {wid: idx for idx, wid in enumerate(sorted(te_dict.keys()))}
        
        new_te_dict = dict()
        for k in te_dict.keys():
            new_te_dict[wid2label_te[k]] = te_dict[k]
    
    # ایجاد datasets با corpus
    data_train = Persian_words(new_tr_dict, oov, FARSI_CORPUS)
    data_test = Persian_words(new_te_dict, oov, FARSI_CORPUS)
    
    print(f"\n{'='*60}")
    print(f"✅ بارگذاری دیتاست فارسی کامل شد")
    print(f"{'='*60}")
    print(f"📊 Train: {len(data_train)} نویسنده")
    print(f"📊 Test:  {len(data_test)} نویسنده")
    print(f"📊 الفبا: {num_classes} کاراکتر (A-Z + a-z + _)")
    print(f"📊 Corpus: {len(FARSI_CORPUS)} کلمه فارسی")
    print(f"📊 OOV Mode: {'✅' if oov else '❌'}")
    print(f"{'='*60}\n")
    
    return data_train, data_test

# ==================== تست ====================
if __name__ == "__main__":
    print("\n🧪 تست دیتالودر فارسی\n")
    
    # تست با OOV=False
    print("=" * 60)
    print("تست 1: OOV = False")
    print("=" * 60)
    data_train, data_test = loadData(oov=False)
    sample = data_train[0]
    print(f"   Domain: {sample[0]}")
    print(f"   Writer ID: {sample[1]}")
    print(f"   Num images: {sample[3].shape}")
    print(f"   label_xt shape: {sample[7].shape}")
    print(f"   label_xt: {sample[7]}")
    print(f"   label_xt_swap shape: {sample[8].shape}")
    print(f"   label_xt_swap: {sample[8]}")
    
    # دیکد کردن label
    def decode_label(label_array):
        result = []
        for idx in label_array:
            if idx == tokens["GO_TOKEN"]:
                result.append("<GO>")
            elif idx == tokens["END_TOKEN"]:
                result.append("<END>")
            elif idx == tokens["PAD_TOKEN"]:
                result.append("<PAD>")
            else:
                result.append(index2letter[idx - num_tokens])
        return "".join(result)
    
    print(f"   label_xt decoded: {decode_label(sample[7])}")
    print(f"   label_xt_swap decoded: {decode_label(sample[8])}")
    print(f"\n✅ تست موفق!")


🔤 الفبا: 53 کاراکتر
   A-Z: 26 حرف بزرگ
   a-z: 26 حرف کوچک
   _  : 1 underscore
   مجموع: 53 = 53
✅ بارگذاری corpus: 0 کلمه

🧪 تست دیتالودر فارسی

تست 1: OOV = False

✅ بارگذاری دیتاست فارسی کامل شد
📊 Train: 350 نویسنده
📊 Test:  150 نویسنده
📊 الفبا: 53 کاراکتر (A-Z + a-z + _)
📊 Corpus: 0 کلمه فارسی
📊 OOV Mode: ❌

   Domain: src
   Writer ID: 0
   Num images: (15, 64, 216)
   label_xt shape: (12,)
   label_xt: [ 0 22 10 11 20 22  7  7 16 22 10  1]
   label_xt_swap shape: (12,)
   label_xt_swap: [ 0 22 10 11 20 35  7  7 16 22 10  1]
   label_xt decoded: <GO>THIRTEENTH<END>
   label_xt_swap decoded: <GO>THIRgEENTH<END>

✅ تست موفق!


In [23]:
import os

# بررسی نمونه فایل‌ها
data_dir = './datasets/aban/words'
all_files = sorted([f for f in os.listdir(data_dir) if f.endswith('.png')])

print("🔍 نمونه‌ای از نام فایل‌ها:")
print("="*80)
for i, fname in enumerate(all_files[:20]):
    print(f"{i+1:3d}. {fname}")

print("\n" + "="*80)
print(f"📊 تعداد کل: {len(all_files)} فایل")

# بررسی نمونه از GT
gt_file = './Groundtruth_farsi/gan.aban.tr_va.gt.filter27'
with open(gt_file, 'r', encoding='utf-8') as f:
    lines = [line.strip() for line in f if line.strip()][:20]

print("\n🔍 نمونه‌ای از خطوط GT:")
print("="*80)
for i, line in enumerate(lines):
    print(f"{i+1:3d}. {line}")


🔍 نمونه‌ای از نام فایل‌ها:
  1. ID0000100_p4_B20.png
  2. ID0000100_p4_B21.png
  3. ID0000100_p4_B22.png
  4. ID0000100_p4_B23.png
  5. ID0000100_p4_B24.png
  6. ID0000100_p4_B25.png
  7. ID0000100_p4_B26.png
  8. ID0000100_p4_B27.png
  9. ID0000100_p4_B28.png
 10. ID0000100_p4_B29.png
 11. ID0000100_p4_B30.png
 12. ID0000100_p4_B31.png
 13. ID0000100_p4_B32.png
 14. ID0000100_p4_B33.png
 15. ID0000100_p4_B34.png
 16. ID0000100_p4_B35.png
 17. ID0000100_p4_B36.png
 18. ID0000100_p4_B37.png
 19. ID0000100_p4_B38.png
 20. ID0000100_p4_B39.png

📊 تعداد کل: 62481 فایل

🔍 نمونه‌ای از خطوط GT:
  1. 0000476,ID0000476_p4_B43 TOMAN
  2. 0000476,ID0000476_p4_B52 THOUSAND
  3. 0000476,ID0000476_p5_B45 SEVENHUNDRED
  4. 0000476,ID0000476_p4_B40 TEL
  5. 0000476,ID0000476_p5_B12 NINETEENTH
  6. 0000476,ID0000476_p5_B41 SEVENTY
  7. 0000476,ID0000476_p6_B14 ORDIBEHESHT
  8. 0000476,ID0000476_p4_B49 FIRST
  9. 0000476,ID0000476_p6_B18 ESFAND
 10. 0000476,ID0000476_p6_B21 AZAR
 11. 0000476,ID0000476_p

In [3]:
import cv2
import os
import numpy as np
from glob import glob

img_folder = "./datasets/iam/words"

# پیدا کردن تمام فایل‌های .png
image_files = glob(os.path.join(img_folder, "*.png"))[:10000]  # اول 100 تا رو چک می‌کنیم

heights = []
widths = []

for img_path in image_files:
    img = cv2.imread(img_path, 0)
    if img is not None:
        h, w = img.shape
        heights.append(h)
        widths.append(w)

if heights:
    print(f"📊 آمار سایز تصاویر IAM:")
    print(f"   ارتفاع - میانگین: {np.mean(heights):.1f}, کمینه: {min(heights)}, بیشینه: {max(heights)}")
    print(f"   عرض - میانگین: {np.mean(widths):.1f}, کمینه: {min(widths)}, بیشینه: {max(widths)}")
    print(f"\n💡 پیشنهاد:")
    print(f"   IMG_HEIGHT = {int(np.median(heights))}")
    print(f"   IMG_WIDTH = {int(np.percentile(widths, 95))}")  # 95 درصد تصاویر پوشش داده شن
else:
    print("⚠️ هیچ تصویری پیدا نشد!")



📊 آمار سایز تصاویر IAM:
   ارتفاع - میانگین: 70.6, کمینه: 1, بیشینه: 304
   عرض - میانگین: 157.2, کمینه: 1, بیشینه: 1934

💡 پیشنهاد:
   IMG_HEIGHT = 68
   IMG_WIDTH = 374


In [4]:
import re

# خواندن GT و استخراج تمام کاراکترها
unique_chars = set()
with open("Groundtruth_farsi/gan.aban.tr_va.gt.filter27", "r", encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split(' ', 2)
        if len(parts) >= 3:
            text = parts[2]
            unique_chars.update(text)

# مرتب‌سازی
sorted_chars = sorted(unique_chars)

print(f"📊 تعداد کاراکترهای یونیک: {len(sorted_chars)}")
print(f"🔤 کاراکترها:")
for char in sorted_chars:
    print(f"   '{char}' (Unicode: U+{ord(char):04X})")

# مقایسه با الفبای کد قبلیت
your_alphabet = list("آابپتثجچحخدذرزژسشصضطظعغفقکگلمنوهیئء ‌")
print(f"\n📝 الفبای کد قبلی شما: {len(your_alphabet)} کاراکتر")

missing_in_code = set(sorted_chars) - set(your_alphabet)
extra_in_code = set(your_alphabet) - set(sorted_chars)

if missing_in_code:
    print(f"\n⚠️ کاراکترهای موجود در GT ولی نیستن در الفبا:")
    for char in missing_in_code:
        print(f"   '{char}' (U+{ord(char):04X})")

if extra_in_code:
    print(f"\n❓ کاراکترهای موجود در الفبا ولی نیستن در GT:")
    for char in extra_in_code:
        print(f"   '{char}' (U+{ord(char):04X})")


📊 تعداد کاراکترهای یونیک: 4
🔤 کاراکترها:
   'ا' (Unicode: U+0627)
   'ر' (Unicode: U+0631)
   'م' (Unicode: U+0645)
   'ه' (Unicode: U+0647)

📝 الفبای کد قبلی شما: 37 کاراکتر

❓ کاراکترهای موجود در الفبا ولی نیستن در GT:
   'ج' (U+062C)
   'آ' (U+0622)
   'ز' (U+0632)
   'ش' (U+0634)
   'ظ' (U+0638)
   'ئ' (U+0626)
   'و' (U+0648)
   'ط' (U+0637)
   'پ' (U+067E)
   'غ' (U+063A)
   'ح' (U+062D)
   'ل' (U+0644)
   'ء' (U+0621)
   'ف' (U+0641)
   'د' (U+062F)
   'چ' (U+0686)
   'ق' (U+0642)
   'ت' (U+062A)
   'ن' (U+0646)
   'ع' (U+0639)
   'ی' (U+06CC)
   'ض' (U+0636)
   'ذ' (U+0630)
   ' ' (U+0020)
   'ث' (U+062B)
   'ب' (U+0628)
   '‌' (U+200C)
   'گ' (U+06AF)
   'ک' (U+06A9)
   'س' (U+0633)
   'ژ' (U+0698)
   'ص' (U+0635)
   'خ' (U+062E)
